# Finite Tutoring Task and Identifiability

Reproduce the finite positive-support task validation, its single C3 prior revision, and the synthetic C2 rule-identifiability check. This task permits at most 20 information-acquisition queries per trial and uses exact discrete Bayesian updates. It is not the earlier continuous Gaussian model, and its cap does not retroactively alter those comparisons. Expected utility is the performance criterion; allocation resemblance does not identify a person's cognitive rule.

The baseline and revised configurations are retained because both were reported. C3 changes from [60, 60] to [50, 70]; this is not a pure mean-gap manipulation because finite prior moments change too.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)

In [ ]:
import json
import shlex

RUN = False
SCALE = "smoke"  # "full" reproduces the reported numerical scale.
assert SCALE in ("smoke", "full")
OUTPUT = ROOT / "results" / "finite_reproduction" / SCALE
if RUN:
    OUTPUT.mkdir(parents=True, exist_ok=True)

def execute(arguments):
    command = [sys.executable, *map(str, arguments)]
    print(shlex.join(command))
    if RUN:
        subprocess.run(command, cwd=ROOT, check=True)

def show_json(relative):
    path = OUTPUT / relative
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2))
    else:
        print("Not generated:", path.name)

## Baseline Five-Condition Validation

Full: 1,200 episodes per condition, 6,000 total; 1,001-point allocation grid with prespecified 2,001-point sensitivity checks. Smoke: 30 per condition for wiring only. Repeated one-step sampling is not a globally optimal metalevel solution.

In [ ]:
execute(["scripts/validate_tutoring_design.py", "--conditions-file", "configs/tutoring_conditions_baseline.json",
         "--stage", "precision" if SCALE == "full" else "smoke", "--sensitivity",
         "--output-dir", OUTPUT / "baseline"])

In [ ]:
show_json("baseline/summary.json")

## Single C3 Revision

The fixed confirmation uses 1,200 episodes per condition and an independent root seed. Screening uses 30 per condition. The baseline file and numerical-engine hashes are checked. The public hashes reflect the cleaned scientific configuration and documentation-only source cleanup.

In [ ]:
revision_stages = ("screening", "confirmation") if SCALE == "full" else ("screening",)
for stage in revision_stages:
    execute(["scripts/validate_tutoring_design_revision.py", "--stage", stage,
             "--output-dir", OUTPUT / "revised" / stage])

In [ ]:
for stage in revision_stages:
    show_json(f"revised/{stage}/selection_gate.json")

## C2 Identifiability

Full mode runs screening and independent confirmation of the prespecified response-rule/noise cells with synthetic participants. Group responsiveness and individual classification are separate outcomes. Failed or ambiguous cells remain in the output; success is not human construct validation.

In [ ]:
identifiability_stages = ("screening", "confirmation") if SCALE == "full" else ("smoke",)
for stage in identifiability_stages:
    execute(["scripts/validate_allocation_identifiability.py", "--conditions", "configs/tutoring_conditions_baseline.json",
             "--stage", stage, "--output-dir", OUTPUT / "identifiability" / stage])

In [ ]:
for stage in identifiability_stages:
    show_json(f"identifiability/{stage}/summary.json")

## Finite Versus Gaussian Reference

This deterministic audit checks the affine GPA mapping and the effects of finite support. The Gaussian reference is not a rerun of the older solver.

In [ ]:
execute(["scripts/audit_tutoring_model_bridge.py", "--design", "configs/tutoring_conditions.json",
         "--output-dir", OUTPUT / "model_bridge"])

In [ ]:
show_json("model_bridge/model_bridge_summary.json")

## Interpretation

Use the complete outputs, not only the selected C3 result. Synthetic recovery and numerical checks do not validate recruitment, human behavior, or a global metalevel optimum. Output directories are exclusive: choose a new path for another execution.